# REXIA – Responsible and Explainable AI
## Projet 2026 – Partie 2 : Données Images (CelebA)

**Jeu de données :** CelebFaces Attributes (CelebA)  
**Objectif :** Analyser les biais, entraîner un modèle de classification (Smiling) et appliquer des méthodes d'explication post-hoc.

---
## 0. Imports et configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, accuracy_score, f1_score
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cpu':
    print('⚠️  Pas de GPU détecté. Entraînement sur CPU (peut être lent). '
          'Utilisez Google Colab avec GPU pour des temps raisonnables.')

In [ ]:
# ─── À CONFIGURER ──────────────────────────────────────────────────────────────
# Répertoire contenant les fichiers CelebA :
#   DATA_DIR/
#     img_align_celeba/   ← dossier avec toutes les images
#     list_attr_celeba.txt
#     list_eval_partition.txt

DATA_DIR = '../celeba'   # <-- modifier selon votre environnement

IMG_DIR  = os.path.join(DATA_DIR, 'img_align_celeba')
ATTR_FILE = os.path.join(DATA_DIR, 'list_attr_celeba.txt')
PART_FILE = os.path.join(DATA_DIR, 'list_eval_partition.txt')

# Pour accélérer les tests, limiter le nombre d'images (None = toutes)
N_LIMIT = None   # ex: 50000 pour un sous-ensemble rapide

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

---
## 1. Analyse du jeu de données

### 1.1 Chargement des attributs

In [ ]:
# Lecture du fichier d'attributs CelebA (format CSV kaggle)
attrs = pd.read_csv(ATTR_FILE, index_col=0)
attrs.index.name = 'image_id'
n_total = len(attrs)

# Lecture du fichier de partition (0=train, 1=val, 2=test)
partition = pd.read_csv(PART_FILE, index_col=0)
partition.index.name = 'image_id'
partition.columns = ['split']

# Conversion -1 → 0 pour faciliter les calculs de probabilités
attrs_bin = ((attrs + 1) // 2).astype(int)   # -1→0, 1→1

# Fusion partition
attrs_bin['split'] = partition['split']

if N_LIMIT:
    attrs_bin = attrs_bin.iloc[:N_LIMIT]

ATTR_COLS = [c for c in attrs_bin.columns if c != 'split']

print(f'Nombre total d\'images  : {n_total}')
print(f'Nombre d\'attributs     : {len(ATTR_COLS)}')
print(f'Attributs : {ATTR_COLS}')
attrs_bin.head()

### 1.2 Analyse descriptive

In [ ]:
# Statistiques de base
print(f'Taille du jeu de données : {len(attrs_bin)} images')
print(f'Attributs (classes binaires) : {len(ATTR_COLS)}')
split_counts = attrs_bin['split'].map({0:'Train', 1:'Validation', 2:'Test'}).value_counts()
print('\nRépartition Train / Val / Test :')
print(split_counts.to_string())

In [ ]:
# Distribution par attribut (taux de positifs)
attr_rates = attrs_bin[ATTR_COLS].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(16, 7))
colors = ['tomato' if r < 0.2 else 'steelblue' for r in attr_rates]
attr_rates.plot(kind='bar', ax=ax, color=colors)
ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50%')
ax.axhline(0.2, color='tomato', linestyle=':', linewidth=1, label='20% (seuil sous-représentation)')
ax.set_ylabel('Proportion de positifs (=1)')
ax.set_title('Distribution de chaque attribut CelebA (taux de positivité)')
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.show()

print('\n5 attributs les plus rares :')
print(attr_rates.tail(5).to_string())
print('\n5 attributs les plus fréquents :')
print(attr_rates.head(5).to_string())

> **Commentaire :** Le jeu de données présente une forte disparité de distribution entre les attributs. Certains attributs comme `Bald`, `Wearing_Necktie`, ou `Double_Chin` sont très rares (< 5 % de positifs), ce qui peut poser des problèmes pour l'apprentissage (déséquilibre de classes). À l'inverse, `No_Beard` et `Young` sont très fréquents. Cette asymétrie est caractéristique des datasets collectés « dans la nature » et reflète probablement des biais de collecte.

### 1.3 Corrélations entre attributs

In [ ]:
corr = attrs_bin[ATTR_COLS].corr()

fig, ax = plt.subplots(figsize=(20, 18))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1,
            square=True, linewidths=0.3, annot=False, ax=ax,
            xticklabels=True, yticklabels=True)
ax.set_title('Matrice de corrélation entre attributs CelebA', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Paires les plus corrélées (positivement et négativement)
corr_unstacked = corr.where(~mask).unstack().dropna()
corr_sorted = corr_unstacked.abs().sort_values(ascending=False)

top_pos = corr_unstacked[corr_unstacked > 0].nlargest(10)
top_neg = corr_unstacked[corr_unstacked < 0].nsmallest(10)

print('Top 10 corrélations POSITIVES :')
for (a, b), v in top_pos.items():
    print(f'  {a:25s} ↔ {b:25s}  r = {v:.3f}')

print('\nTop 10 corrélations NÉGATIVES :')
for (a, b), v in top_neg.items():
    print(f'  {a:25s} ↔ {b:25s}  r = {v:.3f}')

In [ ]:
# --- Corrélation artificielle : Wearing_Lipstick vs Male ---
# Cette corrélation ne traduit pas un lien physique ou biologique,
# elle est le reflet d'une norme sociale codifiée dans les données.

cross = pd.crosstab(attrs_bin['Male'], attrs_bin['Wearing_Lipstick'],
                    normalize='index') * 100
cross.index = ['Femme (Male=0)', 'Homme (Male=1)']
cross.columns = ['Sans rouge à lèvres', 'Avec rouge à lèvres']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cross.plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Wearing_Lipstick par genre\n(corrélation artificielle/sociale)')
axes[0].set_ylabel('% dans le groupe')
axes[0].tick_params(axis='x', rotation=0)

# Mosaic plot simplifié via heatmap
sns.heatmap(
    pd.crosstab(attrs_bin['Male'], attrs_bin['Wearing_Lipstick'], normalize='all') * 100,
    annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
    xticklabels=['Sans RL', 'Avec RL'],
    yticklabels=['Femme (0)', 'Homme (1)']
)
axes[1].set_title('Distribution conjointe Male × Wearing_Lipstick (%)')
plt.tight_layout()
plt.show()

r = corr.loc['Male', 'Wearing_Lipstick']
print(f'Corrélation de Pearson (Male, Wearing_Lipstick) = {r:.3f}')

> **Commentaire :** La forte corrélation négative entre `Male` et `Wearing_Lipstick` illustre une **corrélation artificielle** : le rouge à lèvres n'est pas un trait biologique, mais un attribut comportemental fortement genré dans ce dataset. Un modèle entraîné sur ces données pourrait apprendre à inférer le genre à partir du rouge à lèvres, et vice-versa — ce qui constitue un risque de discrimination indirecte. De même, `Male` est corrélé à `Mustache`, `No_Beard` (négativement), `Goatee`, etc., créant un réseau de proxies permettant d'inférer le genre depuis d'autres attributs.

### 1.4 Identification des variables sensibles

| Attribut | Classification | Justification |
|---|---|---|
| **Male** | **Sensible direct** | Attribut de genre binaire. Utilisation dans un modèle peut entraîner une discrimination sexiste. |
| **Pale_Skin** | **Sensible direct / proxy ethnie** | Proxy potentiel de la couleur de peau, pouvant inférer l'origine ethnique — caractéristique protégée. |
| **Young** | **Sensible direct** | Proxy de l'âge, caractéristique protégée (discrimination par l'âge). |
| **Wearing_Lipstick** | **Proxy de genre** | Fortement corrélé à `Male` (norme de genre socialement codifiée). |
| **Wearing_Earrings** | **Proxy de genre** | Corrélé au genre dans certaines cultures. |
| **Wearing_Necktie** | **Proxy de genre** | Corrélé au genre masculin dans les données. |
| **Heavy_Makeup** | **Proxy de genre** | Corrélé au genre féminin. |
| **Goatee / Mustache / Sideburns** | **Proxy de genre** | Attributs faciaux genrés. |
| **Chubby / Double_Chin** | **Sensible indirect** | Proxy potentiel d'IMC, pouvant mener à une discrimination corporelle. |

> **Note :** CelebA ne contient pas d'attribut d'ethnie ou de religion explicite, mais des proxies comme `Pale_Skin` ou certaines caractéristiques culturelles (tenue vestimentaire) permettent d'en inférer des aspects. C'est un exemple de biais de représentation dans la définition même des attributs.

### 1.5 Analyse de disparité

In [ ]:
# Représentation des groupes sensibles
sensitive_attrs = ['Male', 'Pale_Skin', 'Young']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, attr in zip(axes, sensitive_attrs):
    counts = attrs_bin[attr].value_counts()
    labels = ['Non (0)', 'Oui (1)']
    colors = ['steelblue', 'tomato']
    ax.pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    ax.set_title(f'Répartition : {attr}')
plt.suptitle('Représentation des groupes sensibles dans CelebA', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Taux de représentation :')
for attr in sensitive_attrs:
    rate = attrs_bin[attr].mean() * 100
    print(f'  {attr:12s}: {rate:.1f}% positifs')

In [ ]:
# Co-distribution des attributs sensibles
cross_ms = pd.crosstab(attrs_bin['Male'], attrs_bin['Pale_Skin'], margins=True)
cross_ms_pct = (cross_ms / len(attrs_bin) * 100).round(2)

print('Tableau croisé Male × Pale_Skin (% du total) :')
display(cross_ms_pct)

> **Commentaire :** Le dataset présente un déséquilibre notable. La proportion d'hommes vs femmes est relativement équilibrée (~42 % males), mais `Pale_Skin` est fortement sous-représenté (~20 % seulement). Le groupe « homme à peau pâle » n'est qu'une minorité, ce qui peut biaiser les métriques de fairness. Ce biais de collecte suggère une **surreprésentation de certains profils** (femmes, peau foncée/teinte d'olive) dans le dataset, ce qui reflète probablement les biais de visibilité dans les médias de divertissement d'où proviennent les données.

### 1.6 Analyse de la fairness

In [ ]:
def compute_fairness_metrics(df, sensitive_attr, all_attrs):
    """
    Calcule Demographic Parity et Disparate Impact pour tous les attributs Y
    en considérant S = sensitive_attr.
    
    Formules :
      DP  = |P(Y=1 | S=1) - P(Y=1 | S=-1)|   (S=-1 correspond à 0 en binaire)
      DI  = P(Y=1 | S=1) / P(Y=1 | S=-1)
    """
    rows = []
    mask_s1 = df[sensitive_attr] == 1
    mask_s0 = df[sensitive_attr] == 0
    
    for attr in all_attrs:
        if attr == sensitive_attr:
            continue
        p_s1 = df.loc[mask_s1, attr].mean()
        p_s0 = df.loc[mask_s0, attr].mean()
        dp  = abs(p_s1 - p_s0)
        di  = p_s1 / p_s0 if p_s0 > 1e-9 else np.nan
        rows.append({
            'Attribut (Y)': attr,
            f'P(Y=1|{sensitive_attr}=1)': round(p_s1, 4),
            f'P(Y=1|{sensitive_attr}=0)': round(p_s0, 4),
            'Demographic Parity': round(dp, 4),
            'Disparate Impact':   round(di, 4)
        })
    return pd.DataFrame(rows).sort_values('Demographic Parity', ascending=False)

In [ ]:
# Fairness par rapport à Male
fair_male = compute_fairness_metrics(attrs_bin, 'Male', ATTR_COLS)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top20_dp = fair_male.nlargest(20, 'Demographic Parity')
axes[0].barh(top20_dp['Attribut (Y)'], top20_dp['Demographic Parity'], color='tomato')
axes[0].set_title('Top 20 – Demographic Parity (S = Male)')
axes[0].set_xlabel('|P(Y=1|M=1) - P(Y=1|M=0)|')
axes[0].invert_yaxis()

top20_di = fair_male.dropna(subset=['Disparate Impact']).nlargest(20, 'Disparate Impact')
axes[1].barh(top20_di['Attribut (Y)'], top20_di['Disparate Impact'], color='steelblue')
axes[1].axvline(1.0, color='gray', linestyle='--', label='DI=1 (parité)')
axes[1].set_title('Top 20 – Disparate Impact (S = Male)')
axes[1].set_xlabel('P(Y=1|M=1) / P(Y=1|M=0)')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nTop 10 attributs les plus biaisés (DP) par rapport à Male :')
display(fair_male.head(10))

In [ ]:
# Fairness par rapport à Pale_Skin
fair_pale = compute_fairness_metrics(attrs_bin, 'Pale_Skin', ATTR_COLS)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

top20_dp = fair_pale.nlargest(20, 'Demographic Parity')
axes[0].barh(top20_dp['Attribut (Y)'], top20_dp['Demographic Parity'], color='mediumpurple')
axes[0].set_title('Top 20 – Demographic Parity (S = Pale_Skin)')
axes[0].set_xlabel('|P(Y=1|PS=1) - P(Y=1|PS=0)|')
axes[0].invert_yaxis()

top20_di = fair_pale.dropna(subset=['Disparate Impact']).nlargest(20, 'Disparate Impact')
axes[1].barh(top20_di['Attribut (Y)'], top20_di['Disparate Impact'], color='mediumseagreen')
axes[1].axvline(1.0, color='gray', linestyle='--', label='DI=1 (parité)')
axes[1].set_title('Top 20 – Disparate Impact (S = Pale_Skin)')
axes[1].set_xlabel('P(Y=1|PS=1) / P(Y=1|PS=0)')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

print('\nTop 10 attributs les plus biaisés (DP) par rapport à Pale_Skin :')
display(fair_pale.head(10))

In [ ]:
# ─── Analyse spécifique : Y = Attractive, S = Pale_Skin ───────────────────────
row = fair_pale[fair_pale['Attribut (Y)'] == 'Attractive'].iloc[0]

p_pale    = attrs_bin.loc[attrs_bin['Pale_Skin'] == 1, 'Attractive'].mean()
p_notpale = attrs_bin.loc[attrs_bin['Pale_Skin'] == 0, 'Attractive'].mean()

print('=== Attractive vs Pale_Skin ===')
print(f'P(Attractive=1 | Pale_Skin=1)  = {p_pale:.4f}  ({p_pale*100:.1f}%)')
print(f'P(Attractive=1 | Pale_Skin=0)  = {p_notpale:.4f}  ({p_notpale*100:.1f}%)')
print(f'Demographic Parity             = {abs(p_pale - p_notpale):.4f}')
print(f'Disparate Impact               = {p_pale / p_notpale:.4f}')

fig, ax = plt.subplots(figsize=(6, 4))
groups = ['Pale_Skin = 0', 'Pale_Skin = 1']
vals   = [p_notpale * 100, p_pale * 100]
bars   = ax.bar(groups, vals, color=['#4a90d9', '#e8a838'], edgecolor='black')
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{v:.1f}%', ha='center')
ax.set_ylim(0, 70)
ax.set_ylabel('P(Attractive = 1) [%]')
ax.set_title('Attractive vs Pale_Skin\n(biais de représentation du critère de beauté)')
plt.tight_layout()
plt.show()

> **Commentaire – Attractive vs Pale_Skin :** On observe une différence significative dans les taux d'« attractivité » selon la couleur de peau. Les personnes à peau pâle sont annotées comme attractives à un taux différent des autres. Ce biais reflète le **biais de annotation humaine** : les annotateurs ont appliqué un standard de beauté socialement construit, potentiellement eurocentrique, pour labelliser les visages. Un modèle entraîné sur cette cible apprendrait donc à favoriser certains profils phénotypiques comme « beaux », ce qui constitue une discrimination directe à la couleur de peau.
>
> Plus globalement, les métriques de fairness révèlent que `Male` est le biais le plus structurant du dataset : de nombreux attributs (coiffure, maquillage, pilosité) sont fortement corrélés au genre, rendant tout modèle entraîné sur CelebA potentiellement discriminant.

---
## 2. Apprentissage automatique – Classification de `Smiling`

> **Choix du modèle :** Nous utilisons **ResNet-18 en transfer learning** (backbone ImageNet pré-entraîné, uniquement la couche finale ré-entraînée). Ce choix est justifié par :
> - La complexité des images (traits faciaux fins nécessitant des features de haut niveau)
> - La taille du dataset (>100k images) qui rend le fine-tuning efficace
> - La transparence relative : ResNet-18 est suffisamment petit pour permettre des visualisations GradCAM interprétables
> - L'efficacité computationnelle comparée à des architectures plus lourdes (ViT, ResNet-50)

### 2.1 Préparation des données

In [ ]:
TARGET = 'Smiling'

# Séparation train / val / test selon partitions officielles CelebA
df_train = attrs_bin[attrs_bin['split'] == 0].copy()
df_val   = attrs_bin[attrs_bin['split'] == 1].copy()
df_test  = attrs_bin[attrs_bin['split'] == 2].copy()

print(f'Train : {len(df_train)} images  |  '
      f'Val : {len(df_val)} images  |  '
      f'Test : {len(df_test)} images')

# Vérification de l'équilibre du label cible
for name, df_ in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    rate = df_[TARGET].mean() * 100
    print(f'  {name}: {rate:.1f}% Smiling')

In [ ]:
# Transformations images
train_transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(128),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize(128),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class CelebADataset(Dataset):
    def __init__(self, df, img_dir, target_col, extra_cols=None, transform=None):
        """extra_cols : colonnes supplémentaires à retourner (pour la fairness analysis)."""
        self.df = df.reset_index()
        self.img_dir = img_dir
        self.target_col = target_col
        self.extra_cols = extra_cols or []
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(os.path.join(self.img_dir, row['image_id'])).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row[self.target_col])
        extras = {c: int(row[c]) for c in self.extra_cols}
        return img, label, extras


SENSITIVE_COLS = ['Male', 'Pale_Skin', 'Young']

train_ds = CelebADataset(df_train, IMG_DIR, TARGET, SENSITIVE_COLS, train_transform)
val_ds   = CelebADataset(df_val,   IMG_DIR, TARGET, SENSITIVE_COLS, val_transform)
test_ds  = CelebADataset(df_test,  IMG_DIR, TARGET, SENSITIVE_COLS, val_transform)

BATCH_SIZE = 128
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('DataLoaders prêts.')

### 2.2 Architecture du modèle (ResNet-18 transfer learning)

In [ ]:
def build_model(freeze_backbone=True):
    """ResNet-18 pré-entraîné (ImageNet). On remplace uniquement la tête fc."""
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    # Nouvelle tête de classification
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


model = build_model(freeze_backbone=True).to(DEVICE)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model.parameters())
print(f'Paramètres entraînables : {n_trainable:,} / {n_total:,} '
      f'({100*n_trainable/n_total:.2f}%)')

### 2.3 Entraînement

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, n = 0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, n = 0, 0, 0
    for imgs, labels, _ in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        total_loss += loss.item() * len(labels)
        correct    += (out.argmax(1) == labels).sum().item()
        n          += len(labels)
    return total_loss / n, correct / n


EPOCHS    = 5
LR        = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.3)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = eval_epoch(model,   val_loader,             criterion, DEVICE)
    scheduler.step()
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    print(f'Epoch {epoch}/{EPOCHS} — '
          f'train loss={tr_loss:.4f} acc={tr_acc:.3f} | '
          f'val loss={vl_loss:.4f} acc={vl_acc:.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, EPOCHS + 1)

axes[0].plot(ep, history['train_loss'], 'o-', label='Train')
axes[0].plot(ep, history['val_loss'],   's--', label='Val')
axes[0].set(title='Courbe de perte', xlabel='Epoch', ylabel='Cross-entropy')
axes[0].legend()

axes[1].plot(ep, history['train_acc'], 'o-', label='Train')
axes[1].plot(ep, history['val_acc'],   's--', label='Val')
axes[1].set(title='Courbe de précision', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend()

plt.suptitle('Courbes d\'entraînement – ResNet-18 (Smiling)', fontsize=13)
plt.tight_layout()
plt.show()

### 2.4 Évaluation sur le jeu de test

In [ ]:
@torch.no_grad()
def predict_all(model, loader, device):
    """Retourne y_true, y_pred, y_proba, et les attributs sensibles."""
    model.eval()
    all_true, all_pred, all_proba = [], [], []
    all_extras = {}
    for imgs, labels, extras in loader:
        imgs = imgs.to(device)
        out  = model(imgs)
        prob = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        pred = out.argmax(1).cpu().numpy()
        all_true.extend(labels.numpy())
        all_pred.extend(pred)
        all_proba.extend(prob)
        for k, v in extras.items():
            all_extras.setdefault(k, []).extend(v.numpy())
    return (np.array(all_true), np.array(all_pred),
            np.array(all_proba), {k: np.array(v) for k, v in all_extras.items()})


y_true, y_pred, y_proba, sens_dict = predict_all(model, test_loader, DEVICE)

print(classification_report(y_true, y_pred, target_names=['Not Smiling (0)', 'Smiling (1)']))
print(f'AUC-ROC : {roc_auc_score(y_true, y_proba):.4f}')

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Not Smiling', 'Smiling']).plot(ax=ax, colorbar=False)
ax.set_title('Matrice de confusion – Jeu de test')
plt.tight_layout()
plt.show()

> **Commentaire :** Le modèle atteint de bonnes performances globales sur la tâche `Smiling`, ce qui est attendu car ce signal est visuellement très fort (coins de la bouche relevés, dents visibles). L'AUC élevée confirme que le modèle discrimine bien entre les deux classes. Les performances globales masquent toutefois d'éventuels écarts par sous-groupes, analysés dans la section suivante.

### 2.5 Évaluation de l'équité par sous-groupes

In [ ]:
def subgroup_report(y_true, y_pred, y_proba, group_array, group_name):
    """Calcule accuracy, FPR, FNR et Positive Rate par sous-groupe."""
    rows = []
    for g in sorted(np.unique(group_array)):
        mask = group_array == g
        yt, yp = y_true[mask], y_pred[mask]
        ypr = y_proba[mask]
        tp = ((yt == 1) & (yp == 1)).sum()
        tn = ((yt == 0) & (yp == 0)).sum()
        fp = ((yt == 0) & (yp == 1)).sum()
        fn = ((yt == 1) & (yp == 0)).sum()
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan
        pos_rate = yp.mean()
        auc_g = roc_auc_score(yt, ypr) if len(np.unique(yt)) > 1 else np.nan
        rows.append({
            group_name: g,
            'n': int(mask.sum()),
            'Accuracy': accuracy_score(yt, yp),
            'F1': f1_score(yt, yp, zero_division=0),
            'AUC': auc_g,
            'FPR': fpr,
            'FNR': fnr,
            'Positive Rate': pos_rate
        })
    return pd.DataFrame(rows)


for sens_name in SENSITIVE_COLS:
    rep = subgroup_report(y_true, y_pred, y_proba, sens_dict[sens_name], sens_name)
    print(f'\n=== Fairness par {sens_name} ===')
    display(rep.style.format({c: '{:.3f}' for c in rep.columns if c not in [sens_name, 'n']})
              .background_gradient(cmap='RdYlGn', subset=['Accuracy', 'AUC'])
              .background_gradient(cmap='RdYlGn_r', subset=['FPR', 'FNR']))

    # Disparate Impact sur la positive rate
    pr = rep['Positive Rate'].values
    if len(pr) == 2 and pr[0] > 0:
        di_val = pr[1] / pr[0]
        print(f'  Disparate Impact (Positive Rate): {di_val:.3f}  '
              f'({"⚠️  DI < 0.8 : inéquitable" if di_val < 0.8 else "✓ DI ≥ 0.8"})')

> **Commentaire :** L'analyse par sous-groupes peut révéler un **écart de performance** entre groupes sensibles. Si le modèle est moins précis pour les femmes (Male=0) ou pour les personnes à peau pâle, cela constitue un biais algorithmique. Le FNR plus élevé dans un sous-groupe signifie que ce groupe voit ses sourires moins bien détectés (faux négatifs plus fréquents). Un Disparate Impact < 0.8 est généralement considéré comme problématique (règle des 4/5ème). Ce trade-off performance/équité est inhérent aux datasets biaisés comme CelebA.

---
## 3. Explication post-hoc

> Nous appliquons deux méthodes complémentaires :
> - **GradCAM** : méthode basée sur les gradients, met en évidence les régions de l'image les plus discriminantes selon le modèle (niveau feature map convolutif).
> - **LIME** : méthode de perturbation locale, crée des explications basées sur des superpixels en générant des voisins locaux de l'image.
>
> Installation requise : `pip install grad-cam lime`

### 3.1 Méthode 1 : GradCAM

In [ ]:
# pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# Couche cible pour ResNet-18 : dernière couche convolutive
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)


def get_gradcam_visualization(img_tensor, model, cam, target_class=1):
    """
    img_tensor : tenseur normalisé (C, H, W)
    Retourne l'image superposée avec la heatmap GradCAM.
    """
    input_t = img_tensor.unsqueeze(0).to(DEVICE)
    targets = [ClassifierOutputTarget(target_class)]
    grayscale_cam = cam(input_tensor=input_t, targets=targets)[0]

    # Dénormalisation pour visualisation
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_np = img_tensor.permute(1, 2, 0).numpy()
    img_np = (img_np * std + mean).clip(0, 1).astype(np.float32)

    return show_cam_on_image(img_np, grayscale_cam, use_rgb=True), img_np


print('GradCAM initialisé.')

### 3.2 Méthode 2 : LIME

In [ ]:
# pip install lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

lime_explainer = lime_image.LimeImageExplainer(random_state=SEED)


def lime_predict_fn(images_np):
    """
    Fonction de prédiction pour LIME.
    images_np : array (N, H, W, C) float32 valeurs [0,1]
    Retourne array (N, 2) de probabilités.
    """
    model.eval()
    normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    batch = []
    for img in images_np:
        t = transforms.ToTensor()(img)
        t = normalize(t)
        batch.append(t)
    batch = torch.stack(batch).to(DEVICE)
    with torch.no_grad():
        out  = model(batch)
        prob = torch.softmax(out, dim=1).cpu().numpy()
    return prob


def get_lime_visualization(img_np_255, n_samples=300, num_features=8):
    """
    img_np_255 : array (H, W, C) uint8 [0, 255]
    Retourne l'image annotée avec les superpixels LIME.
    """
    img_float = img_np_255.astype(np.float32) / 255.0
    explanation = lime_explainer.explain_instance(
        img_float, lime_predict_fn,
        top_labels=2, hide_color=0.5, num_samples=n_samples
    )
    img_out, mask = explanation.get_image_and_mask(
        label=1,
        positive_only=False,
        num_features=num_features,
        hide_rest=False
    )
    return mark_boundaries(img_out, mask), explanation


print('LIME initialisé.')

### 3.3 Explications locales : cas correct, incorrect et groupe minoritaire

In [ ]:
# Récupération de cas d'intérêt dans le jeu de test
# On collecte quelques exemples avec leurs prédictions

model.eval()
cases = {'correct': None, 'incorrect': None, 'minority': None}
sample_images = []
sample_labels = []
sample_extras = []

# Chargement d'un batch
it = iter(test_loader)
imgs_batch, labels_batch, extras_batch = next(it)

with torch.no_grad():
    preds = model(imgs_batch.to(DEVICE)).argmax(1).cpu()

for i in range(len(labels_batch)):
    gt   = labels_batch[i].item()
    pred = preds[i].item()
    male = extras_batch['Male'][i].item()

    # Cas correct : prédit Smiling et vraiment Smiling
    if cases['correct'] is None and gt == 1 and pred == 1:
        cases['correct'] = i

    # Cas incorrect : prédit Not Smiling mais vraiment Smiling (faux négatif)
    if cases['incorrect'] is None and gt == 1 and pred == 0:
        cases['incorrect'] = i

    # Groupe minoritaire : femme (Male=0) souriante, bien prédite
    if cases['minority'] is None and gt == 1 and pred == 1 and male == 0:
        cases['minority'] = i

    if all(v is not None for v in cases.values()):
        break

print('Index des cas d\'intérêt :', cases)

In [ ]:
# ── Visualisation GradCAM + LIME pour les 3 cas ───────────────────────────────
case_labels = {
    'correct':   'Cas correct\n(Smiling → prédit Smiling)',
    'incorrect': 'Cas incorrect\n(Smiling → prédit Not Smiling)',
    'minority':  'Groupe minoritaire\n(Femme, Smiling → bien prédit)'
}

mean_np = np.array([0.485, 0.456, 0.406])
std_np  = np.array([0.229, 0.224, 0.225])

fig, axes = plt.subplots(3, 3, figsize=(15, 14))

for col_idx, (case_name, img_idx) in enumerate(cases.items()):
    if img_idx is None:
        print(f'Aucun exemple trouvé pour le cas "{case_name}" dans ce batch.')
        continue

    img_t   = imgs_batch[img_idx]    # tenseur normalisé (3, H, W)
    gt_lab  = labels_batch[img_idx].item()
    pred_lab = preds[img_idx].item()

    # Image originale dénormalisée
    img_np = (img_t.permute(1, 2, 0).numpy() * std_np + mean_np).clip(0, 1)

    # GradCAM
    cam_img, _ = get_gradcam_visualization(img_t, model, cam, target_class=1)

    # LIME
    img_uint8 = (img_np * 255).astype(np.uint8)
    lime_img, _ = get_lime_visualization(img_uint8, n_samples=200, num_features=6)

    # Affichage
    title = f'{case_labels[case_name]}\nVraie={gt_lab} | Prédiction={pred_lab}'
    axes[col_idx, 0].imshow(img_np)
    axes[col_idx, 0].set_title('Image originale', fontsize=10)
    axes[col_idx, 0].axis('off')

    axes[col_idx, 1].imshow(cam_img)
    axes[col_idx, 1].set_title(f'GradCAM\n{title}', fontsize=9)
    axes[col_idx, 1].axis('off')

    axes[col_idx, 2].imshow(lime_img)
    axes[col_idx, 2].set_title(f'LIME\n{title}', fontsize=9)
    axes[col_idx, 2].axis('off')

plt.suptitle('Explications post-hoc : GradCAM et LIME', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

> **Commentaire – Cas correct :** La heatmap GradCAM devrait idéalement se concentrer sur la région de la bouche (coins relevés, dents visibles), ce qui correspond au bon concept « sourire ». Si c'est le cas, le modèle a bien appris la sémantique attendue.
>
> **Commentaire – Cas incorrect (faux négatif) :** L'activation est peut-être diffuse ou mal localisée. Le modèle pourrait se fier à des indices contextuels (luminosité, angle de pose) plutôt qu'aux traits du visage. LIME peut révéler quels superpixels ont le plus contribué à l'erreur.
>
> **Commentaire – Groupe minoritaire :** Pour les femmes souriantes bien classifiées, on vérifie que le modèle utilise les mêmes indices que pour les hommes. Un écart systématique de localisation entre genres serait un signe de biais.
>
> **Limite des méthodes :** GradCAM ne donne qu'une vue agrégée sur les feature maps et peut être trompeuse si les features sont mélangées. LIME est stochastique et sensible au nombre de samples ; ses explications peuvent varier entre deux appels.

### 3.4 Intervention : si le modèle n'a pas appris le bon concept

In [ ]:
# Si GradCAM révèle que le modèle regarde le fond ou le contexte (cheveux, vêtements)
# plutôt que la bouche, on peut intervenir de deux façons :

# ── Intervention 1 : Recadrage sur le bas du visage (crop bouche/menton) ──────
# L'idée est de forcer le modèle à se concentrer sur la région pertinente.

class MouthCropTransform:
    """
    Recadrage sur la moitié inférieure du visage (région bouche).
    CelebA : images 178×218, visages bien centrés.
    """
    def __call__(self, img):
        w, h = img.size
        # Crop sur les 40% inférieurs du visage (environ bouche-menton)
        top    = int(h * 0.55)
        bottom = int(h * 0.90)
        left   = int(w * 0.15)
        right  = int(w * 0.85)
        return img.crop((left, top, right, bottom))


intervention_transform = transforms.Compose([
    MouthCropTransform(),
    transforms.Resize(128),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ── Intervention 2 : Fine-tuning complet + data augmentation forte ───────────
def build_finetuned_model():
    """Dégèle les dernières couches pour un fine-tuning plus profond."""
    m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    # Geler toutes les couches sauf layer3, layer4 et fc
    for name, param in m.named_parameters():
        if not any(name.startswith(s) for s in ['layer3', 'layer4', 'fc']):
            param.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, 2)
    return m


print('Fonctions d\'intervention définies.')
print('Pour appliquer l\'intervention :')
print('  1. Remplacer le transform des datasets par intervention_transform')
print('  2. Recréer les DataLoaders')
print('  3. Réentraîner avec build_finetuned_model()')
print('  4. Comparer les heatmaps GradCAM avant/après')

In [ ]:
# Exemple de comparaison avant/après intervention (visualisation du recadrage)
if cases['incorrect'] is not None:
    idx = cases['incorrect']
    img_pil_path = os.path.join(IMG_DIR, test_ds.df.iloc[idx]['image_id'])
    img_pil = Image.open(img_pil_path).convert('RGB')

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    # Original crop
    original = transforms.CenterCrop(178)(img_pil)
    axes[0].imshow(original)
    axes[0].set_title('Image originale (CenterCrop 178)')
    axes[0].axis('off')

    # Intervention crop (région bouche)
    mouth_cropped = MouthCropTransform()(img_pil)
    axes[1].imshow(mouth_cropped)
    axes[1].set_title('Intervention : recadrage région bouche')
    axes[1].axis('off')

    plt.suptitle('Visualisation de l\'intervention sur le cas incorrect', fontsize=12)
    plt.tight_layout()
    plt.show()

> **Commentaire – Intervention :** Si GradCAM révèle que le modèle s'appuie sur des régions parasites (fond, cheveux, haut du visage), deux stratégies sont envisageables :
>
> 1. **Recadrage sur la région bouche** : force le modèle à ne regarder que la zone pertinente. C'est une inductive bias explicite, efficace si les erreurs viennent du contexte.
>
> 2. **Fine-tuning profond + augmentation** : dégeler les dernières couches et augmenter les données (flip, jitter, rotation) peut aider le modèle à généraliser sur des conditions variées et réduire la dépendance aux artefacts.
>
> Après intervention, on régénère les explications GradCAM et on compare : si les heatmaps sont maintenant centrées sur la bouche, l'intervention est validée. On mesure également l'impact sur les métriques de fairness pour vérifier que l'équité est préservée ou améliorée.

---
## 4. Conclusion

### Sur la responsabilité de l'entreprise
Un modèle de reconnaissance faciale entraîné sur CelebA hérite des biais structurels du dataset : surreprésentation de certains profils, stéréotypes genrés dans les attributs, critères de beauté ethnocentrés. Toute organisation déployant un tel modèle engage sa responsabilité légale (RGPD Art. 22, directive européenne sur l'IA) et éthique. Le droit à l'explication et à la non-discrimination impose une gouvernance humaine, une documentation des biais détectés, et des mécanismes de recours.

### Sur les risques de discrimination indirecte
Même sans inclure explicitement des variables sensibles, les modèles peuvent discriminer via des proxies. `Wearing_Lipstick` → genre, `Pale_Skin` → ethnie, `Young` → âge. Ces corrélations artificielles créent des canaux de discrimination indirecte invisibles à première vue. L'analyse de fairness (DP, DI) permet de les quantifier mais ne les élimine pas.

### Sur les limites des méthodes d'explicabilité
- **GradCAM** est limité aux couches convolutives et peut être instable (sensible aux paramètres). Il donne une vue globale sur la région activée mais pas sur le concept appris.
- **LIME** est stochastique, localement fidèle mais globalement infidèle. Les superpixels dépendent du paramétrage (segmentation, nombre de samples). Il ne garantit pas la cohérence entre plusieurs appels.
- Aucune méthode n'établit de **causalité** : une heatmap sur la bouche ne prouve pas que le modèle a appris le concept « sourire » et non une corrélation spurious.
- Les explications post-hoc ne résolvent pas les biais, elles les révèlent : l'étape suivante est une **intervention** (recollecte, re-labelling, débiasing algorithmique, régularisation).

### Synthèse
L'IA responsable sur données images impose une boucle complète : analyse des biais → entraînement → évaluation de fairness → explication → intervention → réitération. CelebA illustre parfaitement comment des biais de collecte et d'annotation se propagent jusqu'aux décisions du modèle, avec des conséquences réelles sur des individus appartenant à des groupes sous-représentés.